# DLT/SDP Recommendation on basis of scenario

| #  | Source Type       | Load Type                 | Trigger / Timing | Incremental? | **DLT Approach**                      | Recommendation | Why                |
| -- | ----------------- | ------------------------- | ---------------- | ------------ | ------------------------------------- | -------------- | ------------------ |
| 1  | Files (ADLS / S3) | Full Load                 | One-time         | ❌            | `dlt.table + spark.read`              | ⭐              | Simple batch       |
| 2  | Files             | Incremental               | Daily (fixed)    | ✅            | Auto Loader + DLT (trigger once)      | ⭐⭐⭐            | File tracking      |
| 3  | Files             | Incremental + Transform   | Daily            | ✅            | Auto Loader + DLT transforms          | ⭐⭐⭐            | Scalable           |
| 4  | Files             | Incremental               | On arrival       | ✅            | Auto Loader + DLT streaming           | ⭐⭐⭐            | Event-driven       |
| 5  | Files             | Incremental (manual)      | Daily            | ❌            | ❌ Not recommended                     | ❌              | No state           |
| 6  | Delta Table       | Full Load                 | Initial          | ❌            | `dlt.table + spark.read.table`        | ⭐              | Simple overwrite   |
| 7  | Delta Table       | Incremental (Insert only) | Daily            | ✅            | DLT + Delta CDF                       | ⭐⭐⭐            | Reads only changes |
| 8  | Delta Table       | Incremental (Upsert)      | Daily            | ✅            | DLT + CDF + `apply_changes`           | ⭐⭐⭐            | Correct updates    |
| 9  | Delta Table       | Incremental (Insert only) | Near RT          | ✅            | Streaming DLT + CDF                   | ⭐⭐⭐            | Low latency        |
| 10 | Delta Table       | Incremental (Upsert)      | Near RT          | ✅            | Streaming DLT + CDF + `apply_changes` | ⭐⭐⭐            | Exactly-once       |



## 1) Daily Files → Insert at Fixed Time (No Complex Logic)
### Recommended: Auto Loader with trigger(once=True)
Why:
- Incremental file tracking
- Exactly-once
- No full folder scan every day

**Pipeline mode: Triggered (once / availableNow)**

In [0]:
@dlt.table(name="bronze_files_incremental")
def bronze_files_incremental():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .load("abfss://raw@storage.dfs.core.windows.net/data")
    )

## 2) Daily Files → Complex Transformations → Delta
### Recommended: Auto Loader + Spark Transformations
Why:
- Same incremental ingestion
- Transformations run inside micro-batch

**Auto Loader ≠ no transformations**
It’s just the ingestion layer.

In [0]:
@dlt.table(name="silver_files")
def silver_files():
    df = dlt.read_stream("bronze_files_incremental")
    return df.filter("status = 'ACTIVE'").select("id", "amount")

## 3) Incremental Source Delta → Only Inserts (No Updates)
### Recommended: Delta Change Data Feed (CDF)

Why
- No need to scan full table
- Reads only new rows since last version

In [0]:
%sql
-- prerequisite :
ALTER TABLE source_delta_table
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
import dlt

@dlt.table(
    name="silver_delta_insert_only",
    comment="Incremental insert-only load using Delta Change Data Feed"
)
def silver_delta_insert_only():
    return (
        spark.readStream
        .format("delta")
        .option("readChangeFeed", "true")
        .table("source_delta_table")
        .filter("_change_type = 'insert'")     # Filters only newly inserted rows
    )


## 4) Incremental Source Delta → Inserts + Updates
### Recommended: CDF + MERGE

Why
- Handles updates & inserts
- Maintains correctness

In [0]:
%sql
-- prerequisite :
ALTER TABLE source_delta_table
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
# Step 1: Read Delta Change Data Feed
import dlt
@dlt.view(
    name="source_delta_cdf",
    comment="Delta Change Data Feed for incremental upserts"
)
def source_delta_cdf():
    return (
        spark.readStream
        .format("delta")
        .option("readChangeFeed", "true")
        .table("source_delta_table")
        .filter("_change_type IN ('insert', 'update_postimage')")
    )

# Step 2: Apply MERGE using dlt.apply_changes (SCD Type 1)
dlt.apply_changes(
    target="silver_delta_upsert",
    source="source_delta_cdf",
    keys=["id"],
    sequence_by="_commit_version",
    apply_as_deletes="_change_type = 'delete'",
    stored_as_scd_type=1
)


## 5) Daily Files → Insert Based on File Arrival
### Recommended: Auto Loader (Continuous Mode)

Why
- Processes files immediately
- Event-driven
- Best for near real-time

**No scheduling needed, 
Runs continuously**

In [0]:
import dlt

@dlt.table(
    name="bronze_files_streaming",
    comment="Continuous file ingestion using Auto Loader"
)
def bronze_files_streaming():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")   # parquet/csv/json
        .option("cloudFiles.schemaLocation", "/mnt/schema/bronze_files")
        .load("abfss://raw@storage.dfs.core.windows.net/input/")    )


## 6) Streaming from a Delta Table (Real-Time / Near Real-Time)
- Streaming mode automatically handles incremental changes.
- checkpointLocation is mandatory for exactly-once guarantees.
- Use outputMode("append") or update depending on target.

In [0]:
import dlt

@dlt.table(
    name="silver_delta_streaming",
    comment="Real-time streaming from Delta table"
)
def silver_delta_streaming():
    return (
        spark.readStream
        .format("delta")
        .table("source_delta_table")
    )